In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../Data/Raw/Telco_customer_churn.csv')

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nFirst 5 rows:")
df.head()

Shape: (7043, 33)

Columns: ['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']

First 5 rows:


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [2]:
# Check data types and missing values
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nChurn Distribution:")
print(df['Churn Label'].value_counts())
print("\nChurn Rate:", round(df['Churn Label'].value_counts(normalize=True)['Yes'] * 100, 2), "%")

Data Types:
CustomerID            object
Count                  int64
Country               object
State                 object
City                  object
Zip Code               int64
Lat Long              object
Latitude             float64
Longitude            float64
Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Phone Service         object
Multiple Lines        object
Internet Service      object
Online Security       object
Online Backup         object
Device Protection     object
Tech Support          object
Streaming TV          object
Streaming Movies      object
Contract              object
Paperless Billing     object
Payment Method        object
Monthly Charges      float64
Total Charges         object
Churn Label           object
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason          object
dtype: object

Missing Values:


In [4]:
# Fix Total Charges — convert to numeric
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

print("NaN in Total Charges after conversion:", df['Total Charges'].isnull().sum())

# Fix for pandas 3.0 compatibility
df['Total Charges'] = df['Total Charges'].fillna(0)

print("Total Charges dtype now:", df['Total Charges'].dtype)

NaN in Total Charges after conversion: 0
Total Charges dtype now: float64


In [5]:
# Feature Engineering

# 1. Tenure groups
df['Tenure Group'] = pd.cut(df['Tenure Months'],
    bins=[0, 12, 24, 48, 72],
    labels=['New (0-12mo)', 'Growing (1-2yr)', 'Loyal (2-4yr)', 'Champion (4yr+)'])

# 2. Average monthly spend
df['Avg Monthly Spend'] = df['Total Charges'] / (df['Tenure Months'] + 1)

# 3. Service count
service_cols = ['Phone Service', 'Online Security', 'Online Backup',
                'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies']

for col in service_cols:
    df[col] = df[col].replace({'No internet service': 'No', 'No phone service': 'No'})
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df['Service Count'] = df[service_cols].sum(axis=1)

print("New features added:")
print(df[['Tenure Months', 'Tenure Group', 'Avg Monthly Spend', 'Service Count']].head(10))

New features added:
   Tenure Months     Tenure Group  Avg Monthly Spend  Service Count
0              2     New (0-12mo)          36.050000              3
1              2     New (0-12mo)          50.550000              1
2              8     New (0-12mo)          91.166667              4
3             28    Loyal (2-4yr)         105.036207              5
4             49  Champion (4yr+)         100.726000              5
5             10     New (0-12mo)          48.031818              3
6              1     New (0-12mo)          19.825000              2
7              1     New (0-12mo)          10.075000              1
8             47    Loyal (2-4yr)          98.940625              4
9              1     New (0-12mo)          15.100000              1


In [6]:
# Encode binary Yes/No columns to 0 and 1
binary_cols = ['Partner', 'Dependents', 'Paperless Billing', 'Churn Label']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Encode Gender
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

# One-hot encode multi-category columns
df = pd.get_dummies(df, columns=['Contract', 'Internet Service', 'Payment Method'], drop_first=True)

print("Shape after encoding:", df.shape)
print("\nNew columns added:")
print([col for col in df.columns if 'Contract' in col or 'Internet' in col or 'Payment' in col])

Shape after encoding: (7043, 40)

New columns added:
['Contract_One year', 'Contract_Two year', 'Internet Service_Fiber optic', 'Internet Service_No', 'Payment Method_Credit card (automatic)', 'Payment Method_Electronic check', 'Payment Method_Mailed check']


In [ ]:
# Drop columns not needed for analysis or modeling
drop_cols = ['CustomerID', 'Count', 'Country', 'State', 'City', 
             'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Churn Reason']

df_clean = df.drop(columns=drop_cols)

# Save cleaned dataset
df_clean.to_csv('../Data/processed/telco_cleaned.csv', index=False)

print("Cleaned dataset saved!")
print("Final shape:", df_clean.shape)
print("\nColumns:", list(df_clean.columns))

Cleaned dataset saved!
Final shape: (7043, 30)

Columns: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Paperless Billing', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Tenure Group', 'Avg Monthly Spend', 'Service Count', 'Contract_One year', 'Contract_Two year', 'Internet Service_Fiber optic', 'Internet Service_No', 'Payment Method_Credit card (automatic)', 'Payment Method_Electronic check', 'Payment Method_Mailed check']
